# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer Exploration with `mlcroissant`
This notebook provides a step-by-step workflow for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
ds = mlc.Dataset(croissant_url)
md = ds.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Below we list all record sets (`@id`), their fields (`@id`), and columns (`@id`).

In [ ]:
# List all record sets and their fields/columns
record_sets = ds.metadata.recordSet
record_set_ids = []
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    record_set_ids.append(rs['@id'])
    if 'field' in rs:
        print("  Fields:")
        for f in rs['field']:
            print(f"    Field @id: {f['@id']}, name: {f.get('name', '(no name)')}, dataType: {f.get('dataType', 'unknown')}")
            if 'column' in f:
                print("      Columns:")
                for col in f['column']:
                    print(f"        Column @id: {col['@id']}, name: {col.get('name', '(no name)')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here we extract all available record sets for exploration.

In [ ]:
# Extract all available record sets
dataframes = {}
for rs_id in record_set_ids:
    records = list(ds.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Columns for RecordSet {rs_id}:\n{df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for RecordSet {rs_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Here, we'll select a numeric field and a group (categorical) field from the available columns for analysis. Replace the `example_numeric_field_id` and `example_group_field_id` with your chosen `@id` values as listed above.

In [ ]:
# Select a record set for detailed EDA
# Replace these variables with actual @id's from previous cells
example_record_set_id = record_set_ids[0] if record_set_ids else None
numeric_field_id = None
group_field_id = None

if example_record_set_id and example_record_set_id in dataframes:
    print("Available columns for EDA:", dataframes[example_record_set_id].columns.tolist())
    # Attempt to select a numeric and group field by guessing based on column names
    df = dataframes[example_record_set_id]
    # Try to identify numeric fields
    for col in df.columns:
        # Use dtype to select numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Try to identify a grouping (categorical) column
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and (df[col].nunique() < 10):
            group_field_id = col
            break
    threshold = 10
    if numeric_field_id:
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
            print(f"Grouped data by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No numeric fields found for EDA.")
else:
    print("No valid record set found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

We will plot the distribution of the selected numeric field, and if grouping field exists, visualize group comparisons.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if example_record_set_id and numeric_field_id and example_record_set_id in dataframes:
    df = dataframes[example_record_set_id]
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    if group_field_id:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Insufficient numeric and grouping fields for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

* This notebook demonstrated how to load and explore the FAIR² dataset using mlcroissant.
* We reviewed available record sets and fields by their `@id`, loaded them into DataFrames, and executed sample EDA and visualizations.
* The dataset enables stratification analysis of second primary colorectal cancer in survivors, supporting clinical and biomarker research.